## 1. Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
import condpred
from condpred import selfcheck, report
from condpred.config import smoke

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/selfchecks" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

SPEC = {
    "experiment": "00_selfchecks",
    "version": 3,
    "changelog": ["01b", "01b: torch RP parity + timing; cdist compute_mode fix, parity reference on spec definitions"],
    "smoke": SMOKE,
}

import scipy
import sklearn

print("numpy", np.__version__)
print("scipy", scipy.__version__)
print("sklearn", sklearn.__version__)
try:
    import lightgbm as lgb
    print("lightgbm", lgb.__version__)
except ImportError as e:
    raise RuntimeError("lightgbm required for self-checks") from e


## 2. Write spec

In [ ]:
report.write_spec(SPEC, OUT, modules=["condpred.selfcheck", "condpred.rp_batch", "condpred.rp_rqa_stage2"])



## 3. Known-answer tests

In [ ]:
rng = np.random.default_rng(0)
results = selfcheck.run_all(rng)
df = pd.DataFrame(results, columns=["test", "pass", "detail"])
report.save_table(df, OUT, "known_answer")
assert df["pass"].all(), df[~df["pass"]]

## 4. Leakage tests

In [ ]:
rng = np.random.default_rng(4)
n = smoke(800, 300)
x = rng.standard_normal(n)
leaks = selfcheck.run_leakage_suite_backends(x, rng)
ldf = pd.DataFrame(leaks, columns=["pipeline", "pass"])
report.save_table(ldf, OUT, "leakage")
assert ldf["pass"].all(), ldf[~ldf["pass"]]


## 5. Timing table

In [ ]:
Ws = [80, 250] if SMOKE else [80, 250, 500]
rng = np.random.default_rng(1)
from condpred.geometry import Geometry
from condpred.features.window_rqa import window_rqa
from condpred.features.local_rqa import local_rqa

geom = Geometry()
x = rng.standard_normal(smoke(2000, 600))
rows = []
n_rep = smoke(20, 3)
for W in Ws:
    t0 = time.perf_counter()
    for end in range(W, W + n_rep * smoke(10, 20), smoke(10, 20)):
        window_rqa(x, end, W, 3, 1, geom)
        local_rqa(x, end, W, 3, 1, geom)
    ms = (time.perf_counter() - t0) / n_rep * 1000
    rows.append({"W": W, "ms_per_window": ms})
timing = pd.DataFrame(rows)
report.save_table(timing, OUT, "timing")

## 6. RP backend parity

In [ ]:
rng = np.random.default_rng(2)
parity = selfcheck.rp_parity(rng, smoke=SMOKE, device_list=["cpu", "cuda"])
report.save_table(parity, OUT, "parity")
bad = parity.loc[parity["mismatches"] != 0]
assert bad.empty, bad


## 7. Stage-2 line histograms

In [ ]:
stage2 = selfcheck.rp_stage2_parity(np.random.default_rng(5), smoke=SMOKE, device_list=["cpu", "cuda"])
report.save_table(stage2, OUT, "parity_stage2")
bad2 = stage2.loc[stage2["mismatches"] != 0]
assert bad2.empty, bad2


## 8. Backend timing (§01b §7)

In [ ]:
timing_b = selfcheck.timing_backend(np.random.default_rng(3), smoke=SMOKE)
report.save_table(timing_b, OUT, "timing_backend")
cpu_rqa = timing_b["cpu_rqa_ms"].sum()
loc = timing_b["local_rqa_ms"].sum()
torch_rp = timing_b.get("torch_cuda_rp_ms", timing_b["torch_cpu_rp_ms"]).fillna(0).sum()
total = cpu_rqa + loc + torch_rp
need_gpu_hist = total > 0 and (cpu_rqa + loc) / total > 0.5
print("(c)+(d) share", (cpu_rqa + loc) / total if total else 0, "stage2", need_gpu_hist)
